# 3. Skip ahead

You found it. If you already run agents in production, the talk will be familiar: here are six things to build instead, on the same agent, while it runs.

Each one has the *why* (which part of the talk it belongs to), *where* in the code, and *done when*. No solutions, only hints. Pick one, they don't depend on each other.

Everything is in `/workspace/research`. After a change: `start('try', QUESTION)`, then `files()`, `trace()`, `changes()`, `check()` from `workshop.py`. Mentors know this notebook exists: ask them.

In [ ]:
from workshop import start, status, show, log, files, trace, changes, check

QUESTION = ('We sell HR and payroll software to small and mid-size companies in Europe, '
            'like Personio. Brief me for a call with Contentful in 1 hour. Source every claim.')

## 1. Give the fact checker its own model

**Why:** the micro-benchmark. The small model is fine for research but adds unsourced claims when it checks facts. Today all workers share `WORKER_MODEL`.

**Where:** `models.py` (read a third env var, e.g. `FACT_CHECK_MODEL`, build a third `ChatNebius`) and `agents.py` (`fact_checker_subagent["model"]`). `openshell/run.py` only passes the variables listed in `SANDBOX_CONFIG_KEYS` into the sandbox: add yours there.

**Done when:** a run with a small `WORKER_MODEL` and a strong `FACT_CHECK_MODEL` passes `check()` and `changes()` shows real fixes.

## 2. Cap the work: calls and time per brief

**Why:** "One failure costs one brief". A loop or a hung call should end in a clear message to the rep, not a late brief and a big bill.

**Where:** `main.py`, the `agent.invoke(agent_input)` call. LangGraph takes `config={'recursion_limit': N}` and raises `GraphRecursionError` past it. For wall time, run the invoke in a thread and stop waiting after a deadline.

**Done when:** with a tiny limit, the run ends quickly with a short "no brief, prepare by hand" file instead of a stack trace, and `status()` shows it as done, not failed.

## 3. Log tokens per role

**Why:** the economics block. Cost per brief is the metric, but nothing logs tokens today, only model calls.

**Where:** `main.py`. `langchain_core.callbacks.UsageMetadataCallbackHandler` collects `usage_metadata` per model name: pass it in the invoke `config={'callbacks': [handler]}` and write `handler.usage_metadata` to `usage.json` in the run folder. With different models per role, per model is per role.

**Done when:** every run folder has a `usage.json`, and you can say what one brief cost, per role.

In [ ]:
# starter: read usage.json from your latest run once you've added it
import json
from workshop import _done
state = _done('try')
if state:
    print(json.dumps(json.loads((state['run_dir'] / 'usage.json').read_text()), indent=2))

## 4. Treat the form as data

**Why:** the real input. The `message` field is written by the prospect and the agent reads it as part of the request. Dasha's guardrail is one fix; here is another layer.

**Where:** before `start()`: split the CRM record into trusted fields (company, country, size) and untrusted text (message), and pass the message as quoted data with an explicit instruction in `prompts.py` that claims from it must be verified like any other. Or add a cheap pre-check with a small model that flags instructions in the message.

**Done when:** `check('messy')` never shows the planted claim as a fact, on three runs in a row.

## 5. Turn the micro-benchmark into an eval

**Why:** "Quality controls": evaluations gate every change. Three hand-read tests are a start, not a gate.

**Where:** start from `TESTS` and `run_tests()` in `01_token_factory.ipynb`. Grow it to 10+ cases (planted claims, wrong company, number vs source, outdated facts). Replace reading by eye with a judge: the strong model answers PASS or FAIL against each case's rule. Report a pass rate per model.

**Done when:** one cell prints pass rate, time and tokens per model, and you'd trust it to block a prompt change.

## 6. Run five briefs and find the outlier

**Why:** "Keep it working". One run tells you nothing about the distribution. The monitoring table alerts on 2× the usual cost or time: what is usual?

**Where:** `start()` five different prospects, a minute apart. Compare `status()` lines: time, model calls, searches. Open the slowest with `trace()` and find why.

**Done when:** you can name the usual time and calls per brief, and explain the outlier.

In [ ]:
for i, prospect in enumerate(['Contentful', 'Celonis', 'Trade Republic', 'Personio', 'N26']):
    # start them a minute apart: each run refreshes the OpenShell providers when it begins
    print(f"start('b{i}', ...) for {prospect}")